# Process description
main dataset: https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset/data

Based on https://arxiv.org/pdf/1901.09851 - Dataset found in zenodo 

> Papreja, P. (2020) “Playlist2vec: Spotify Million Playlist Dataset”. Zenodo. Available at: https://doi.org/10.5281/zenodo.5002584.


A kaggle username and its PAT is needed to retrieve the data, get yours from https://www.kaggle.com/settings/api then generate a "Legacy API Credentials". This data goes within the .env file.

Before running this notebook, initialize a virtual environment and install requirements:

1. `python -m venv .venv`: to create the virtual environment.
2. `.venv\scripts\activate`: to activate the virtual environment using Windows.
3. `pip install -r requirements.txt`: to install the list of requirements.

In [ ]:
import sqlite3
from contextlib import closing

import kaggle
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from zenodo_get import download

from mysql_to_sqlite import load_mysql_dump_to_sqlite

load_dotenv()

In [ ]:
# kaggle dataset

datasets_path = Path("dataset")
if not (datasets_path / "dataset" / "dataset.csv").exists():
    kaggle.api.dataset_download_files(
        "maharshipandya/-spotify-tracks-dataset", path=datasets_path, unzip=True
    )
df_tracks = pd.read_csv(datasets_path / "dataset.csv")

In [ ]:
# zenodo dataset
# Playlist2vec: Spotify Million Playlist Dataset (10.7GB MySQL dump)
if not (datasets_path / "spotifydbdumpshare.sql").exists():
    download("10.5281/zenodo.5002584", output_dir=datasets_path)

## Loading the MySQL dump into SQLite
The zenodo record ships a MySQL dump (schema + ~10.7 GB of `INSERT` statements) instead of a csv. Rather than standing up a MySQL server, `mysql_to_sqlite.py` stream-parses the dump straight into a local SQLite file (never loading the whole 10.7 GB into memory) and is invoked below.

In [ ]:
sql_files = sorted(datasets_path.glob("*.sql"))
schema_path = next(p for p in sql_files if "schema" in p.name.lower())
data_path = next(p for p in sql_files if p != schema_path)

db_path = datasets_path / "spotify.db"
load_mysql_dump_to_sqlite(schema_path, data_path, db_path)

In [ ]:
# sanity-check the assumption that track.id already equals the id embedded in track.uri
with closing(sqlite3.connect(db_path)) as conn:
    id_check = pd.read_sql_query(
        "SELECT id, uri, substr(uri, 15) AS uri_track_id, id = substr(uri, 15) AS ids_match FROM track LIMIT 5",
        conn,
    )
display(id_check)

In [ ]:
with closing(sqlite3.connect(db_path)) as conn:
    # push kaggle track ids into sqlite so the JOIN narrows the result, instead of pulling the full table
    df_tracks[["track_id"]].to_sql(
        "kaggle_track_ids", conn, index=False, if_exists="replace"
    )

    df_playlists = pd.read_sql_query("SELECT * FROM playlist", conn)
    df_track_playlist = pd.read_sql_query(
        "SELECT * FROM track_playlist1 WHERE playlist_id IN (SELECT id FROM playlist)",
        conn,
    )
    # uri is "spotify:track:<id>" (14-char prefix) - strip it to match kaggle's plain track id
    df_sql_tracks = pd.read_sql_query(
        """
        SELECT t.*, substr(t.uri, 15) AS track_id
        FROM track t
        JOIN kaggle_track_ids k ON substr(t.uri, 15) = k.track_id
        """,
        conn,
    )
    # drop the scratch table now that its data has been pulled into the DataFrame above
    conn.execute("DROP TABLE kaggle_track_ids")

In [ ]:
# I stopped previous cell at 25 minutes without completing - I need to figure out a way to streamline it
df_playlists.info()

In [ ]:
# join datasets - kaggle track features with the sql dump's track/album metadata
df_spotify = df_tracks.merge(
    df_sql_tracks, how="inner", on="track_id", suffixes=("", "_dump")
)
display(df_spotify)

# ML Algorithm Implementation
We must consider sparsity of the dataset - most users will have a small number of interactions with the tracks, and most tracks will have a small number of interactions with users.

We begin analysing 
1. [Matrix Factorization Techniques for Recommender Systems](https://doi.org/10.1109/MC.2009.263)
2. [A Machine Learning Recommender System Based on Collaborative Filtering Using Gaussian Mixture Model Clustering](https://doi.org/10.22541/au.160897179.93005705/v1)
3. [Imputation Techniques in Machine Learning – A Survey](https://doi.org/10.17762/ijritcc.v11i10.8662)
4. [Recommender systems applied to the tourism industry: a literature review](https://doi.org/10.1080/23311975.2024.2367088)

> Algorithm alternatives: SVD, ALS, FM, Item-based k-NN